[![Buka di Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kang-prayit/visi-komputer-kesehatan/blob/main/notebooks/bab09.ipynb)

# Bab 9 — Segmentasi Semantik dengan U-Net: Kepala Janin pada USG (HC18)
**Buku:** *Visi Komputer untuk Kesehatan: Klasifikasi, Regresi, Deteksi, dan Segmentasi Citra Medis dengan Python*
**Penulis:** Prayitno, Karisma Trinanda Putra, Eko Prasetyo, Sunardi, Liliek Triyono

Jalankan di **Google Colab dengan GPU T4**, lalu **Runtime → Run all**.

Isi:
1. **HC18** (CC BY 4.0): mask kepala dibuat dengan mengisi garis elips anotasi; pembagian per subjek sama dengan Bab 7 (seed 42).
2. **U-Net** dari nol: *loss* BCE vs Dice + BCE; dan U-Net dengan *encoder* ResNet-18 pra-latih ImageNet (2 seed).
3. Metrik per kasus pada resolusi asli 800×540: **Dice, IoU, HD95 (mm)**; visualisasi tumpang-susun terbaik/median/terburuk.
4. **Dari mask ke ukuran klinis**: paskan elips → keliling (Ramanujan) × ukuran piksel → HC (mm), dibandingkan dengan HC acuan dan dengan regresi langsung Bab 7.

Hasil → `/content/hasil_bab09/` → `/content/hasil_bab09.zip`. Hanya untuk pembelajaran, **bukan** untuk penggunaan klinis.

## 0 Sel standar: mount Google Drive dan `pastikan_dataset()`

In [ ]:
# ===== SEL STANDAR BUKU: Drive + pastikan_dataset() (sama di semua notebook) =====
import os, sys, time, json, hashlib, shutil, zipfile, re, subprocess
from pathlib import Path
try:
    import requests, certifi
except ImportError:  # sudah ada di Colab; dipasang bila notebook dijalankan di tempat lain
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "requests", "certifi"], check=True)
    import requests, certifi
DI_COLAB = "google.colab" in sys.modules
if DI_COLAB:
    from google.colab import drive
    # Lokasi folder buku di Drive dapat diubah lewat variabel lingkungan BUKU_DRIVE_SUBPATH
    BUKU_DRIVE_SUBPATH = os.environ.get("BUKU_DRIVE_SUBPATH", "MyDrive/Buku Visi Komputer Kesehatan")
    for _coba in range(3):  # popup izin Drive kadang lambat: ulangi, dengan force_remount
        try:
            drive.mount("/content/drive", force_remount=_coba > 0); break
        except Exception as e:
            print("drive.mount gagal (", e, "), mencoba lagi ..."); time.sleep(5)
    AKAR_BUKU = Path("/content/drive") / BUKU_DRIVE_SUBPATH
    LOKAL = Path("/content/data_lokal")
else:  # dijalankan di luar Colab: arahkan ke folder lain lewat variabel lingkungan
    AKAR_BUKU = Path(os.environ.get("BUKU_DRIVE", str(Path.home() / "Buku Visi Komputer Kesehatan")))
    LOKAL = Path(os.environ.get("BUKU_LOKAL", "/tmp/data_lokal"))
DATASET_DIR = AKAR_BUKU / "dataset"
DATASET_DIR.mkdir(parents=True, exist_ok=True); LOKAL.mkdir(parents=True, exist_ok=True)
print("Folder dataset di Drive:", DATASET_DIR)

# Katalog sumber resmi: nama berkas -> [URL, ukuran byte, MD5]. MD5 None = sumber tidak menyediakan
# (MD5 malaria/kvasir dicatat penulis dari unduhan 27-10-2026; bila sumber memperbarui berkas, cek ukuran yang dipakai).
MEDMNIST = {  # Zenodo 10.5281/zenodo.10519652 (CC BY 4.0 kecuali DermaMNIST CC BY-NC 4.0)
    'adrenalmnist3d.npz': ['https://zenodo.org/api/records/10519652/files/adrenalmnist3d.npz/content', 276833, 'bbd3c5a5576322bc4cdfea780653b1ce'],
    'adrenalmnist3d_64.npz': ['https://zenodo.org/api/records/10519652/files/adrenalmnist3d_64.npz/content', 1880591, '17721accfe9fb005146a47d33bc54b2f'],
    'bloodmnist.npz': ['https://zenodo.org/api/records/10519652/files/bloodmnist.npz/content', 35461855, '7053d0359d879ad8a5505303e11de1dc'],
    'bloodmnist_128.npz': ['https://zenodo.org/api/records/10519652/files/bloodmnist_128.npz/content', 569072780, 'adace1e0ed228fccda1f39692059dd4c'],
    'bloodmnist_224.npz': ['https://zenodo.org/api/records/10519652/files/bloodmnist_224.npz/content', 1540731655, 'b718ff6835fcbdb22ba9eacccd7b2601'],
    'bloodmnist_64.npz': ['https://zenodo.org/api/records/10519652/files/bloodmnist_64.npz/content', 156251765, '2b94928a2ae4916078ca51e05b6b800b'],
    'breastmnist.npz': ['https://zenodo.org/api/records/10519652/files/breastmnist.npz/content', 559580, '750601b1f35ba3300ea97c75c52ff8f6'],
    'breastmnist_128.npz': ['https://zenodo.org/api/records/10519652/files/breastmnist_128.npz/content', 10961822, '363e4b3f8d712e9b5de15470a2aaadf1'],
    'breastmnist_224.npz': ['https://zenodo.org/api/records/10519652/files/breastmnist_224.npz/content', 30903564, 'b56378a6eefa9fed602bb16d192d4c8b'],
    'breastmnist_64.npz': ['https://zenodo.org/api/records/10519652/files/breastmnist_64.npz/content', 2846041, '742edef2a1fd1524b2efff4bd7ba9364'],
    'chestmnist.npz': ['https://zenodo.org/api/records/10519652/files/chestmnist.npz/content', 82802576, '02c8a6516a18b556561a56cbdd36c4a8'],
    'chestmnist_128.npz': ['https://zenodo.org/api/records/10519652/files/chestmnist_128.npz/content', 1426132371, 'db107e5590b27930b62dbaf558aebee3'],
    'chestmnist_224.npz': ['https://zenodo.org/api/records/10519652/files/chestmnist_224.npz/content', 3889293042, '45bd33e6f06c3e8cdb481c74a89152aa'],
    'chestmnist_64.npz': ['https://zenodo.org/api/records/10519652/files/chestmnist_64.npz/content', 401604127, '9de6cd0b934ebb5b7426cfba5efbae16'],
    'dermamnist.npz': ['https://zenodo.org/api/records/10519652/files/dermamnist.npz/content', 19725078, '0744692d530f8e62ec473284d019b0c7'],
    'dermamnist_128.npz': ['https://zenodo.org/api/records/10519652/files/dermamnist_128.npz/content', 372579947, '2defd784463fa5243564e855ed717de1'],
    'dermamnist_224.npz': ['https://zenodo.org/api/records/10519652/files/dermamnist_224.npz/content', 1091112502, '8974907d8e169bef5f5b96bc506ae45d'],
    'dermamnist_64.npz': ['https://zenodo.org/api/records/10519652/files/dermamnist_64.npz/content', 100130190, 'b70a2f5635c6199aeaa28c31d7202e1f'],
    'fracturemnist3d.npz': ['https://zenodo.org/api/records/10519652/files/fracturemnist3d.npz/content', 3278419, '6aa7b0143a6b42da40027a9dda61302f'],
    'fracturemnist3d_64.npz': ['https://zenodo.org/api/records/10519652/files/fracturemnist3d_64.npz/content', 26565236, 'f01d7e6316aedf4210da0da5b7437b42'],
    'nodulemnist3d.npz': ['https://zenodo.org/api/records/10519652/files/nodulemnist3d.npz/content', 29299364, '8755a7e9e05a4d9ce80a24c3e7a256f3'],
    'nodulemnist3d_64.npz': ['https://zenodo.org/api/records/10519652/files/nodulemnist3d_64.npz/content', 289765731, 'c47c5b7d457bf6332200d2ea6d64ecd8'],
    'octmnist.npz': ['https://zenodo.org/api/records/10519652/files/octmnist.npz/content', 54938180, 'c68d92d5b585d8d81f7112f81e2d0842'],
    'octmnist_128.npz': ['https://zenodo.org/api/records/10519652/files/octmnist_128.npz/content', 1297741899, '0a97e76651ace45c5d943ee3f65b63ae'],
    'octmnist_224.npz': ['https://zenodo.org/api/records/10519652/files/octmnist_224.npz/content', 3959197818, 'abc493b6d529d5de7569faaef2773ba3'],
    'octmnist_64.npz': ['https://zenodo.org/api/records/10519652/files/octmnist_64.npz/content', 311755777, 'e229e9440236b774d9f0dfef9d07bdaf'],
    'organamnist.npz': ['https://zenodo.org/api/records/10519652/files/organamnist.npz/content', 38247708, '68e3f8846a6bd62f0c9bf841c0d9eacc'],
    'organamnist_128.npz': ['https://zenodo.org/api/records/10519652/files/organamnist_128.npz/content', 707925571, 'eeae80d0a227a8d099027e1b3cfd3b60'],
    'organamnist_224.npz': ['https://zenodo.org/api/records/10519652/files/organamnist_224.npz/content', 1803859544, '50747347e05c87dd3aaf92c49f9f3170'],
    'organamnist_64.npz': ['https://zenodo.org/api/records/10519652/files/organamnist_64.npz/content', 200383427, '2dcccc29b88e6da5a01161ef20cda288'],
    'organcmnist.npz': ['https://zenodo.org/api/records/10519652/files/organcmnist.npz/content', 15526411, 'b9ceb9546e10131b32923c5bbeaea2b1'],
    'organcmnist_128.npz': ['https://zenodo.org/api/records/10519652/files/organcmnist_128.npz/content', 287499235, '773c1f009daa3fe5d9a2a201b2a7ed94'],
    'organcmnist_224.npz': ['https://zenodo.org/api/records/10519652/files/organcmnist_224.npz/content', 760231860, '050f5e875dc056f6768abf94ec9995d1'],
    'organcmnist_64.npz': ['https://zenodo.org/api/records/10519652/files/organcmnist_64.npz/content', 80344589, '3ce34a8724ea6f548e6db4744d03b6a9'],
    'organmnist3d.npz': ['https://zenodo.org/api/records/10519652/files/organmnist3d.npz/content', 32657349, 'a0c5a1ff56af4f155c46d46fbb45a2fe'],
    'organmnist3d_64.npz': ['https://zenodo.org/api/records/10519652/files/organmnist3d_64.npz/content', 361470105, '58a2205adf14a9d0a189cb06dc78bf10'],
    'organsmnist.npz': ['https://zenodo.org/api/records/10519652/files/organsmnist.npz/content', 16528359, '9ab87b696fb54e2a387ebe992d6ed5f1'],
    'organsmnist_128.npz': ['https://zenodo.org/api/records/10519652/files/organsmnist_128.npz/content', 305188399, 'ded0c5fa01a95dc4978b956f613e9b8e'],
    'organsmnist_224.npz': ['https://zenodo.org/api/records/10519652/files/organsmnist_224.npz/content', 802713625, 'b354719e553fbbb2513d5533f52a4cb1'],
    'organsmnist_64.npz': ['https://zenodo.org/api/records/10519652/files/organsmnist_64.npz/content', 85866166, '53a6d115339d874c25e309a994ff46d3'],
    'pathmnist.npz': ['https://zenodo.org/api/records/10519652/files/pathmnist.npz/content', 205615438, 'a8b06965200029087d5bd730944a56c1'],
    'pathmnist_128.npz': ['https://zenodo.org/api/records/10519652/files/pathmnist_128.npz/content', 4257786775, 'ac42d08fb904d92c244187169d1fd1d9'],
    'pathmnist_224.npz': ['https://zenodo.org/api/records/10519652/files/pathmnist_224.npz/content', 12629854322, '2c51a510bcdc9cf8ddb2af93af1eadec'],
    'pathmnist_64.npz': ['https://zenodo.org/api/records/10519652/files/pathmnist_64.npz/content', 1069422170, '55aa9c1e0525abe5a6b9d8343a507616'],
    'pneumoniamnist.npz': ['https://zenodo.org/api/records/10519652/files/pneumoniamnist.npz/content', 4170669, '28209eda62fecd6e6a2d98b1501bb15f'],
    'pneumoniamnist_128.npz': ['https://zenodo.org/api/records/10519652/files/pneumoniamnist_128.npz/content', 75506212, '05b46931834c231683c68f40c47b2971'],
    'pneumoniamnist_224.npz': ['https://zenodo.org/api/records/10519652/files/pneumoniamnist_224.npz/content', 214384716, 'd6a3c71de1b945ea11211b03746c1fe1'],
    'pneumoniamnist_64.npz': ['https://zenodo.org/api/records/10519652/files/pneumoniamnist_64.npz/content', 20606998, '8f4eceb4ccffa70c672198ea285246c6'],
    'retinamnist.npz': ['https://zenodo.org/api/records/10519652/files/retinamnist.npz/content', 3291041, 'bd4c0672f1bba3e3a89f0e4e876791e4'],
    'retinamnist_128.npz': ['https://zenodo.org/api/records/10519652/files/retinamnist_128.npz/content', 46467058, 'e48e916a24454daf90583d4e6efb1a18'],
    'retinamnist_224.npz': ['https://zenodo.org/api/records/10519652/files/retinamnist_224.npz/content', 127992567, 'eae7e3b6f3fcbda4ae613ebdcbe35348'],
    'retinamnist_64.npz': ['https://zenodo.org/api/records/10519652/files/retinamnist_64.npz/content', 13200879, 'afda852cc34dcda56f86ad2b2457dbcc'],
    'synapsemnist3d.npz': ['https://zenodo.org/api/records/10519652/files/synapsemnist3d.npz/content', 38034583, '1235b78a3cd6280881dd7850a78eadb6'],
    'synapsemnist3d_64.npz': ['https://zenodo.org/api/records/10519652/files/synapsemnist3d_64.npz/content', 452767391, '43bd14ebf3af9d3dd072446fedc14d5e'],
    'tissuemnist.npz': ['https://zenodo.org/api/records/10519652/files/tissuemnist.npz/content', 124962739, 'ebe78ee8b05294063de985d821c1c34b'],
    'tissuemnist_128.npz': ['https://zenodo.org/api/records/10519652/files/tissuemnist_128.npz/content', 1625223910, '61b955355d7425a89687b06cca3ce0c2'],
    'tissuemnist_224.npz': ['https://zenodo.org/api/records/10519652/files/tissuemnist_224.npz/content', 3433703243, 'b077128c4a949f0a4eb01517f9037b9c'],
    'tissuemnist_64.npz': ['https://zenodo.org/api/records/10519652/files/tissuemnist_64.npz/content', 555266078, '123ece2eba09d0aa5d698fda57103344'],
    'vesselmnist3d.npz': ['https://zenodo.org/api/records/10519652/files/vesselmnist3d.npz/content', 398349, 'b41fd4f7e7e2feedddb201585ecafa1b'],
    'vesselmnist3d_64.npz': ['https://zenodo.org/api/records/10519652/files/vesselmnist3d_64.npz/content', 2653988, '6bb274a8846e1097066dcd64e2c4520f']
}
HC18 = {      # Zenodo 10.5281/zenodo.1327317 (CC BY 4.0)
    'test_set.zip': ['https://zenodo.org/api/records/1327317/files/test_set.zip/content', 43518463, '8402af5d137ef40a2888c1011ef3fe7e'],
    'test_set_pixel_size.csv': ['https://zenodo.org/api/records/1327317/files/test_set_pixel_size.csv/content', 9096, '8476dc198cc6542f26c57e87faeee033'],
    'training_set.zip': ['https://zenodo.org/api/records/1327317/files/training_set.zip/content', 132926838, '00eb8198b9a505b2b3a6dfc740382497'],
    'training_set_pixel_size_and_HC.csv': ['https://zenodo.org/api/records/1327317/files/training_set_pixel_size_and_HC.csv/content', 33688, 'c0761518fece2bd2d2ad4218f2cd9777']
}
KATALOG = {
    "medmnist":   {"berkas": MEDMNIST},
    "hc18":       {"berkas": HC18},
    "malaria":    {"berkas": {"cell_images.zip": ["https://data.lhncbc.nlm.nih.gov/public/Malaria/cell_images.zip", 353452851, "2f2e94a91d413e5b0bab6342fddb6de6"]},
                   "isi": {"cell_images.zip": {".png": 27558}}},
    "kvasir_seg": {"berkas": {"kvasir-seg.zip": ["https://datasets.simula.no/downloads/kvasir-seg.zip", 46227172, "6323d9094df93b35d43069a566ee1ca3"]},
                   "isi": {"kvasir-seg.zip": {".jpg": 2000}}},
    "bccd":       {"berkas": {"BCCD_Dataset-master.zip": ["https://github.com/Shenggan/BCCD_Dataset/archive/refs/heads/master.zip", None, None]},
                   "isi": {"BCCD_Dataset-master.zip": {"/JPEGImages/.jpg": 364, "/Annotations/.xml": 364}}},
}
# Rantai sertifikat: server datasets.simula.no tidak mengirim sertifikat perantara (Sectigo).
# Kita TIDAK mematikan verifikasi TLS; sertifikat perantara publik diambil dari crt.sh dan ditambahkan ke bundel CA.
CRT_PERANTARA = {"datasets.simula.no": "https://crt.sh/?d=4267304690"}

def md5_berkas(p, blok=8 << 20):
    h = hashlib.md5()
    with open(p, "rb") as f:
        for b in iter(lambda: f.read(blok), b""): h.update(b)
    return h.hexdigest()

def _cek_isi_zip(p, aturan):
    nama = zipfile.ZipFile(p).namelist()
    for pola, n in aturan.items():
        folder, ext = (pola.rsplit("/", 1)[0] + "/", pola.rsplit("/", 1)[1]) if "/" in pola else ("", pola)
        jml = sum(1 for x in nama if x.lower().endswith(ext) and folder in x and "__MACOSX" not in x)
        if jml != n: return f"isi zip {pola}: {jml} berkas, seharusnya {n}"
    return None

def berkas_lengkap(nama, fn):
    """Cek berkas di Drive: ada, ukuran sesuai, MD5 sesuai (MD5 dihitung sekali lalu dicatat di <fn>.md5ok)."""
    url, ukuran, md5 = KATALOG[nama]["berkas"][fn]
    p = DATASET_DIR / nama / fn
    if not p.exists(): return False, "belum ada"
    if ukuran and p.stat().st_size != ukuran: return False, f"ukuran {p.stat().st_size} B, seharusnya {ukuran} B"
    tanda = p.with_name(p.name + ".md5ok")
    if md5 and not (tanda.exists() and tanda.read_text().strip() == md5):
        m = md5_berkas(p)
        if m != md5: return False, f"MD5 {m} tidak cocok ({md5})"
        tanda.write_text(md5)
    aturan = KATALOG[nama].get("isi", {}).get(fn)
    if aturan:
        tanda_isi = p.with_name(p.name + ".isiok")
        if not tanda_isi.exists():
            salah = _cek_isi_zip(p, aturan)
            if salah: return False, salah
            tanda_isi.write_text(json.dumps(aturan))
    return True, "lengkap"

def _bundel_ca(host):
    import certifi, requests
    pem = requests.get(CRT_PERANTARA[host], timeout=60).text
    assert "BEGIN CERTIFICATE" in pem
    b = LOKAL / f"ca_{host}.pem"; b.write_text(open(certifi.where()).read() + "\n" + pem); return str(b)

def unduh(url, tujuan):
    import requests
    from urllib.parse import urlparse
    tmp = LOKAL / "_unduh" / tujuan.name; tmp.parent.mkdir(parents=True, exist_ok=True)
    host = urlparse(url).hostname; verify = True
    for coba in range(2):
        try:
            with requests.get(url, stream=True, timeout=120, verify=verify) as r:
                r.raise_for_status(); total = int(r.headers.get("content-length", 0)); n = 0; t0 = time.time()
                with open(tmp, "wb") as f:
                    for b in r.iter_content(4 << 20):
                        f.write(b); n += len(b)
                        if time.time() - t0 > 15: print(f"   ... {n/1e6:.0f}/{total/1e6:.0f} MB"); t0 = time.time()
            break
        except requests.exceptions.SSLError:
            if host in CRT_PERANTARA and verify is True:
                print("   rantai sertifikat server tidak lengkap; menambahkan sertifikat perantara resmi ke bundel CA")
                verify = _bundel_ca(host)
            else:
                raise
    tujuan.parent.mkdir(parents=True, exist_ok=True)
    shutil.copyfile(tmp, tujuan)          # salin ke Drive (permanen)
    return tmp                            # salinan lokal tetap ada untuk dipakai sesi ini

def pastikan_dataset(nama, berkas=None):
    """Pastikan dataset ada & lengkap di Drive; bila belum, unduh dari sumber resmi ke Drive. Kembalikan folder Drive-nya."""
    daftar = berkas or list(KATALOG[nama]["berkas"])
    for fn in daftar:
        ok, alasan = berkas_lengkap(nama, fn)
        if ok:
            print(f"Dataset {nama}/{fn} ditemukan di Drive ({(DATASET_DIR/nama/fn).stat().st_size/1e6:.1f} MB).")
            continue
        print(f"Dataset {nama}/{fn} belum ada atau tidak lengkap ({alasan}); mengunduh dari sumber resmi...")
        t0 = time.time(); unduh(KATALOG[nama]["berkas"][fn][0], DATASET_DIR / nama / fn)
        for t in (".md5ok", ".isiok"):
            q = DATASET_DIR / nama / (fn + t)
            if q.exists(): q.unlink()
        ok, alasan = berkas_lengkap(nama, fn)
        if not ok: raise RuntimeError(f"{nama}/{fn} gagal diverifikasi setelah diunduh: {alasan}")
        print(f"   selesai {time.time()-t0:.0f} s, tersimpan di Drive dan terverifikasi.")
    return DATASET_DIR / nama

def ekstrak_lokal(nama, fn):
    """Ekstrak arsip dari Drive ke disk lokal Colab (/content), sekali per sesi.
    Alasan: membaca ribuan berkas kecil langsung dari Drive (FUSE) sangat lambat; disk lokal jauh lebih cepat."""
    tujuan = LOKAL / nama; tanda = tujuan / (".ekstrak_" + fn)
    if tanda.exists(): print(f"{fn} sudah diekstrak di {tujuan}"); return tujuan
    t0 = time.time(); src = LOKAL / "_unduh" / fn
    if not src.exists() or src.stat().st_size != (DATASET_DIR / nama / fn).stat().st_size:
        src.parent.mkdir(parents=True, exist_ok=True); shutil.copyfile(DATASET_DIR / nama / fn, src)
    with zipfile.ZipFile(src) as z: z.extractall(tujuan)
    tanda.write_text("ok"); print(f"{fn} diekstrak ke {tujuan} dalam {time.time()-t0:.0f} s"); return tujuan

def berkas_medmnist(flag, size):
    return f"{flag}.npz" if size == 28 else f"{flag}_{size}.npz"
# ===== akhir sel standar =====

## 9.0 Lingkungan

In [ ]:
import os, sys, time, json, platform, subprocess, contextlib, shutil, random
from pathlib import Path
os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")   # perlu untuk algoritme deterministik di GPU
T_MULAI = time.perf_counter()
SMOKE = os.environ.get("BAB09_SMOKE") == "1"          # uji cepat penulis di luar Colab (CPU, subset kecil); di Colab = False
OUT = Path(os.environ.get("BAB09_OUT", "/content/hasil_bab09"))
if OUT.exists(): shutil.rmtree(OUT)
OUT.mkdir(parents=True, exist_ok=True)

WAKTU = {}
@contextlib.contextmanager
def catat_waktu(nama):
    t0 = time.perf_counter()
    yield
    WAKTU[nama] = round(time.perf_counter() - t0, 2)
    print(f"[waktu] {nama}: {WAKTU[nama]:.1f} s")

def jalankan(cmd):
    try:
        return subprocess.run(cmd, shell=True, capture_output=True, text=True, timeout=60).stdout
    except Exception as e:
        return f"(gagal: {e})"

with catat_waktu("0_info_lingkungan"):
    smi = jalankan("nvidia-smi")
    print(smi if smi.strip() else "nvidia-smi tidak tersedia (tidak ada GPU?). Di Colab: Runtime > Change runtime type > T4 GPU.")
    info_cpu = jalankan("lscpu | grep -E 'Model name|^CPU\\(s\\)'"); info_mem = jalankan("free -h | head -2"); info_disk = jalankan("df -h /content 2>/dev/null || df -h /")
    print(info_cpu, info_mem, info_disk)
    (OUT / "info_lingkungan.txt").write_text(f"Python: {sys.version}\nPlatform: {platform.platform()}\n\n{smi}\n{info_cpu}\n{info_mem}\n{info_disk}")
print("SMOKE =", SMOKE, "| OUT =", OUT)

In [ ]:
with catat_waktu("1_pip_install"):
    jalankan('pip install -q "medmnist==3.0.2"')
import numpy as np, pandas as pd, torch, torch.nn as nn, torch.nn.functional as F
import matplotlib, matplotlib.ticker; import matplotlib.pyplot as plt
import torchvision, sklearn, medmnist
from medmnist import INFO
plt.rcParams.update({"figure.dpi": 110, "savefig.dpi": 200, "font.size": 9})
DEV = torch.device("cuda" if torch.cuda.is_available() else "cpu")
VERSI = {"python": platform.python_version(), "torch": torch.__version__, "torchvision": torchvision.__version__,
         "numpy": np.__version__, "scikit-learn": sklearn.__version__, "medmnist": medmnist.__version__,
         "cuda": torch.version.cuda, "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"}
print(json.dumps(VERSI, indent=1))
def npz_medmnist(flag, size):
    fn = berkas_medmnist(flag, size); folder = pastikan_dataset("medmnist", [fn]); return np.load(folder / fn)
SEED = 42
def seed_semua(seed=SEED):
    """Seed Python, NumPy, PyTorch (CPU+GPU) dan minta algoritme deterministik."""
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True; torch.backends.cudnn.benchmark = False
    torch.use_deterministic_algorithms(True, warn_only=True)
seed_semua()
# ---------- Perangkat latih bersama (sama di notebook bab 04 dst.) ----------
MEAN_IN = torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1); SD_IN = torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1)
AMP = DEV.type == "cuda"

def ke_tensor_uint8(X):
    """(N,H,W) atau (N,H,W,C) uint8 -> tensor (N,C,H,W) uint8; disimpan di GPU bila ada (data kecil)."""
    t = torch.as_tensor(X)
    t = t[:, None] if t.ndim == 3 else t.permute(0, 3, 1, 2)
    return t.contiguous().to(DEV)

def augmentasi_gpu(x, flip_h=False, flip_v=False, rot90=False, rot_derajat=0.0, geser=0.0, kecerahan=0.1, kontras=0.1, warna=0.0):
    """x float (N,C,H,W) di [0,1]. Semua acak per sampel kecuali rot90 (per batch)."""
    n = x.shape[0]; d = x.device
    if flip_h: x = torch.where((torch.rand(n, device=d) < 0.5)[:, None, None, None], x.flip(3), x)
    if flip_v: x = torch.where((torch.rand(n, device=d) < 0.5)[:, None, None, None], x.flip(2), x)
    if rot90: x = torch.rot90(x, int(torch.randint(0, 4, (1,)).item()), (2, 3))
    if rot_derajat or geser:
        a = (torch.rand(n, device=d) * 2 - 1) * rot_derajat * np.pi / 180
        t = (torch.rand(n, 2, device=d) * 2 - 1) * geser * 2
        th = torch.stack([torch.stack([a.cos(), -a.sin(), t[:, 0]], 1), torch.stack([a.sin(), a.cos(), t[:, 1]], 1)], 1)
        x = F.grid_sample(x, F.affine_grid(th, x.shape, align_corners=False), padding_mode="border", align_corners=False)
    if warna and x.shape[1] == 3: x = x * (1 + (torch.rand(n, 3, 1, 1, device=d) * 2 - 1) * warna)
    m = x.mean((1, 2, 3), keepdim=True)
    x = ((x - m) * (1 + (torch.rand(n, 1, 1, 1, device=d) * 2 - 1) * kontras) + m) * (1 + (torch.rand(n, 1, 1, 1, device=d) * 2 - 1) * kecerahan)
    return x.clamp(0, 1)

class ResNet18(nn.Module):
    """ResNet-18 torchvision; masukan 1 atau 3 kanal [0,1], opsional diperbesar ke `uk` piksel, normalisasi ImageNet."""
    def __init__(self, k, pretrain=True, uk=None, p_drop=0.0):
        super().__init__(); self.uk = uk
        w = None
        if pretrain:
            w = torchvision.models.ResNet18_Weights.IMAGENET1K_V1
            try: self.net = torchvision.models.resnet18(weights=w)
            except Exception as e:
                if not SMOKE: raise
                print("SMOKE: bobot ImageNet gagal diunduh, pakai bobot acak:", e); self.net = torchvision.models.resnet18(weights=None)
        else: self.net = torchvision.models.resnet18(weights=None)
        self.net.fc = nn.Sequential(nn.Dropout(p_drop), nn.Linear(self.net.fc.in_features, k))
    def forward(self, x):
        if x.shape[1] == 1: x = x.repeat(1, 3, 1, 1)
        if self.uk and x.shape[-1] != self.uk: x = F.interpolate(x, size=(self.uk, self.uk), mode="bilinear", align_corners=False)
        return self.net((x - MEAN_IN.to(x.device)) / SD_IN.to(x.device))

@torch.no_grad()
def logit_semua(model, X, bs=512, praproses=None):
    model.eval(); out = []
    for i in range(0, len(X), bs):
        x = X[i:i + bs].to(DEV).float().div(255)
        if praproses: x = praproses(x)
        with torch.autocast("cuda", dtype=torch.float16, enabled=AMP): out.append(model(x).float())
    return torch.cat(out)

def latih(nama, model, X, Y, Xv, Yv, fungsi_loss, skor_val, maks_epoch=30, patience=5, aug=None, lr=1e-3, bs=128,
          tahap=None, bobot_sampel=None, praproses=None, seed=SEED, wd=1e-4):
    """Latih dengan AMP; early stopping memantau skor_val(logit_val, y_val) (lebih tinggi lebih baik).
    tahap: daftar (n_epoch, lr, beku_backbone). bobot_sampel: oversampling (sampling dengan pengembalian)."""
    seed_semua(seed); model = model.to(DEV); Yd = torch.as_tensor(Y).to(DEV); Yvd = torch.as_tensor(Yv).to(DEV)
    if SMOKE: maks_epoch, patience = min(maks_epoch, 2), 1
    scaler = torch.amp.GradScaler("cuda", enabled=AMP); log = []; best = (-np.inf, None, 0); ep_g = 0
    g = torch.Generator(device="cpu").manual_seed(seed)
    for n_ep, lr_t, beku in (tahap or [(maks_epoch, lr, False)]):
        if SMOKE: n_ep = min(n_ep, 2)
        for nm, p in model.named_parameters(): p.requires_grad = (not beku) or nm.startswith("net.fc")
        opt = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=lr_t, weight_decay=wd); tunggu = 0
        for ep in range(n_ep):
            model.train(); t0 = time.perf_counter()
            if bobot_sampel is not None: perm = torch.multinomial(torch.as_tensor(bobot_sampel, dtype=torch.float), len(Y), replacement=True, generator=g)
            else: perm = torch.randperm(len(Y), generator=g)
            perm = perm.to(X.device); tl = 0.0; n = 0
            for i in range(0, len(perm), bs):
                b = perm[i:i + bs]; x = X[b].to(DEV).float().div(255)
                if aug: x = augmentasi_gpu(x, **aug)
                if praproses: x = praproses(x)
                with torch.autocast("cuda", dtype=torch.float16, enabled=AMP):
                    lg = model(x)
                loss = fungsi_loss(lg.float(), Yd[b.to(DEV)])
                opt.zero_grad(set_to_none=True); scaler.scale(loss).backward(); scaler.step(opt); scaler.update()
                tl += loss.item() * len(b); n += len(b)
            if DEV.type == "cuda": torch.cuda.synchronize()
            dt = time.perf_counter() - t0; ep_g += 1
            lv = logit_semua(model, Xv, praproses=praproses); vl = fungsi_loss(lv, Yvd).item(); sv = float(skor_val(lv, Yv))
            log.append({"model": nama, "epoch": ep_g, "tahap_beku": beku, "loss_latih": tl / n, "loss_val": vl, "skor_val": sv, "detik_epoch": dt})
            print(f"{nama} ep{ep_g:02d} loss {tl/n:.4f}/{vl:.4f} skor_val {sv:.4f} {dt:.1f}s")
            if sv > best[0]: best = (sv, {k: v.detach().clone() for k, v in model.state_dict().items()}, ep_g); tunggu = 0
            else:
                tunggu += 1
                if not beku and tunggu >= patience: print(f"  early stopping (patience {patience})"); break
    model.load_state_dict(best[1])
    return model, pd.DataFrame(log), best[2]

def n_param(m): return sum(p.numel() for p in m.parameters())

def ci_bootstrap(fn, y, p, n_boot=1000, seed=SEED, strata=True):
    """CI 95 % persentil; resampling bertingkat per kelas agar setiap sampel memuat semua kelas."""
    r = np.random.default_rng(seed); y = np.asarray(y); idx_k = [np.where(y == k)[0] for k in np.unique(y)]; v = []
    for _ in range(n_boot):
        i = np.concatenate([r.choice(ik, len(ik)) for ik in idx_k]) if strata else r.choice(len(y), len(y))
        try: v.append(fn(y[i], p[i]))
        except ValueError: pass
    return float(np.percentile(v, 2.5)), float(np.percentile(v, 97.5))

def gambar_kurva(LOG, judul_skor, path):
    k = len(LOG); fig, ax = plt.subplots(1, k, figsize=(4.2 * k, 3.2), squeeze=False); ax = ax[0]
    for a, lg in zip(ax, LOG):
        a.plot(lg.epoch, lg.loss_latih, "o-", ms=3, label="loss latih"); a.plot(lg.epoch, lg.loss_val, "s-", ms=3, label="loss val")
        a2 = a.twinx(); a2.plot(lg.epoch, lg.skor_val, "^-", ms=3, color="tab:green", label=judul_skor); a2.set_ylabel(judul_skor, color="tab:green")
        eb = int(lg.loc[lg.skor_val.idxmax(), "epoch"]); a.axvline(eb, color="red", ls="--", lw=1, label=f"terbaik ep {eb}")
        if lg.tahap_beku.any(): a.axvspan(0.5, lg[lg.tahap_beku].epoch.max() + 0.5, color="gray", alpha=0.15, label="tahap beku")
        a.xaxis.set_major_locator(matplotlib.ticker.MaxNLocator(integer=True)); a.set_title(lg.model.iloc[0], fontsize=8); a.set_xlabel("epoch"); a.legend(fontsize=6)
    plt.tight_layout(); plt.savefig(path, bbox_inches="tight"); plt.show()
# ---------- Pemuat HC18 (CC BY 4.0; van den Heuvel dkk., 2018) ----------
from PIL import Image
from scipy import ndimage as ndi
from skimage import measure
def isi_mask(outline):
    m = ndi.binary_fill_holes(outline)
    if m.sum() < outline.sum() * 3:                 # garis elips tidak tertutup rapat -> tutup dulu
        m = ndi.binary_fill_holes(ndi.binary_closing(outline, iterations=3))
    return m
def keliling_elips_mm(mask, pixel_mm):
    """Paskan elips (momen orde-2) pada komponen terbesar, keliling Ramanujan, dikali ukuran piksel (mm)."""
    lab = measure.label(mask)
    if lab.max() == 0: return np.nan
    p = max(measure.regionprops(lab), key=lambda r: r.area); a, b = p.axis_major_length / 2, p.axis_minor_length / 2
    return float(np.pi * (3 * (a + b) - np.sqrt((3 * a + b) * (a + 3 * b))) * pixel_mm)
def muat_hc18(uk, maks=None):
    folder = pastikan_dataset("hc18", ["training_set.zip", "training_set_pixel_size_and_HC.csv"])
    akar = ekstrak_lokal("hc18", "training_set.zip") / "training_set"
    meta = pd.read_csv(folder / "training_set_pixel_size_and_HC.csv"); meta.columns = ["berkas", "pixel_mm", "hc_mm"]
    if maks: meta = meta.iloc[:maks].copy()
    meta["subjek"] = meta.berkas.str.extract(r"^(\d+)")[0]      # 014_HC, 014_2HC, 014_3HC -> subjek 014 (asumsi)
    def baca(fn):
        im = Image.open(akar / fn).convert("L"); w, h = im.size
        m = isi_mask(np.array(Image.open(akar / fn.replace(".png", "_Annotation.png")).convert("L")) > 127)
        return (np.array(im.resize((uk, uk), Image.BILINEAR)), np.array(Image.fromarray(m.astype(np.uint8) * 255).resize((uk, uk), Image.NEAREST)) > 127, w, h)
    from concurrent.futures import ThreadPoolExecutor
    with ThreadPoolExecutor(8) as ex: hasil = list(ex.map(baca, meta.berkas))
    X = np.stack([r[0] for r in hasil]); M = np.stack([r[1] for r in hasil]); meta["lebar"] = [r[2] for r in hasil]; meta["tinggi"] = [r[3] for r in hasil]
    return X, M, meta.reset_index(drop=True), akar
def bagi_subjek(meta, seed=42, uji=0.2, val=0.1):
    from sklearn.model_selection import GroupShuffleSplit
    g = meta.subjek.values; i = np.arange(len(meta))
    a, t = next(GroupShuffleSplit(1, test_size=uji, random_state=seed).split(i, groups=g))
    b, v = next(GroupShuffleSplit(1, test_size=val / (1 - uji), random_state=seed).split(a, groups=g[a]))
    return {"train": a[b], "val": a[v], "test": t}
from scipy.ndimage import distance_transform_edt, binary_erosion

## 9.1 Data dan mask

In [ ]:
UK = 64 if SMOKE else 256
with catat_waktu("2_muat_hc18"):
    XH, MH, META, AKAR = muat_hc18(UK, maks=120 if SMOKE else None)
    SPL = bagi_subjek(META, seed=SEED)
XT = ke_tensor_uint8(XH); MT = torch.as_tensor(MH[:, None]).to(DEV)
print({s: len(i) for s, i in SPL.items()}, "fraksi piksel kepala (latih):", float(MH[SPL["train"]].mean()))
# Batas bawah metode: elips dari mask ACUAN (resolusi asli) vs HC di CSV
def mask_asli(fn): return isi_mask(np.array(Image.open(AKAR / fn.replace(".png", "_Annotation.png")).convert("L")) > 127)
with catat_waktu("3_mask_asli_uji"):
    M_ASLI = {i: mask_asli(META.berkas[i]) for i in SPL["test"]}
HC_GT_ELIPS = np.array([keliling_elips_mm(M_ASLI[i], META.pixel_mm[i]) for i in SPL["test"]])
print("Elips dari mask acuan vs HC CSV: MAE %.2f mm, bias %.2f mm" % (np.nanmean(np.abs(HC_GT_ELIPS - META.hc_mm.values[SPL["test"]])), np.nanmean(HC_GT_ELIPS - META.hc_mm.values[SPL["test"]])))

## 9.2 Arsitektur U-Net
Encoder (turun 4 kali) – *bottleneck* – decoder (naik 4 kali) dengan **skip connection** yang menyambungkan fitur encoder ke decoder
pada resolusi yang sama. Varian kedua memakai blok ResNet-18 pra-latih sebagai encoder.

In [ ]:
def blok(ci, co): return nn.Sequential(nn.Conv2d(ci, co, 3, padding=1, bias=False), nn.BatchNorm2d(co), nn.ReLU(inplace=True),
                                       nn.Conv2d(co, co, 3, padding=1, bias=False), nn.BatchNorm2d(co), nn.ReLU(inplace=True))
class UNet(nn.Module):
    def __init__(self, c=32):
        super().__init__(); ch = [c, 2 * c, 4 * c, 8 * c, 16 * c]
        self.enc = nn.ModuleList([blok(1, ch[0])] + [blok(ch[i], ch[i + 1]) for i in range(4)])
        self.up = nn.ModuleList([nn.ConvTranspose2d(ch[i + 1], ch[i], 2, stride=2) for i in reversed(range(4))])
        self.dec = nn.ModuleList([blok(2 * ch[i], ch[i]) for i in reversed(range(4))]); self.akhir = nn.Conv2d(ch[0], 1, 1)
    def forward(self, x):
        skip = []
        for i, e in enumerate(self.enc):
            x = e(x if i == 0 else F.max_pool2d(x, 2));
            if i < 4: skip.append(x)
        for u, d, s in zip(self.up, self.dec, reversed(skip)): x = d(torch.cat([u(x), s], 1))
        return self.akhir(x)
class UNetResNet18(nn.Module):
    def __init__(self, pretrain=True):
        super().__init__()
        try: r = torchvision.models.resnet18(weights=torchvision.models.ResNet18_Weights.IMAGENET1K_V1 if pretrain else None)
        except Exception as e:
            if not SMOKE: raise
            print("SMOKE: bobot acak", e); r = torchvision.models.resnet18(weights=None)
        self.l0 = nn.Sequential(r.conv1, r.bn1, r.relu); self.l1 = nn.Sequential(r.maxpool, r.layer1); self.l2, self.l3, self.l4 = r.layer2, r.layer3, r.layer4
        self.d4 = blok(512 + 256, 256); self.d3 = blok(256 + 128, 128); self.d2 = blok(128 + 64, 64); self.d1 = blok(64 + 64, 64); self.d0 = blok(64 + 1, 32)
        self.akhir = nn.Conv2d(32, 1, 1)
    def forward(self, x):
        x0 = x; x = (x.repeat(1, 3, 1, 1) - MEAN_IN.to(x.device)) / SD_IN.to(x.device)
        f0 = self.l0(x); f1 = self.l1(f0); f2 = self.l2(f1); f3 = self.l3(f2); f4 = self.l4(f3)
        up = lambda a, b: F.interpolate(a, size=b.shape[-2:], mode="bilinear", align_corners=False)
        y = self.d4(torch.cat([up(f4, f3), f3], 1)); y = self.d3(torch.cat([up(y, f2), f2], 1)); y = self.d2(torch.cat([up(y, f1), f1], 1))
        y = self.d1(torch.cat([up(y, f0), f0], 1)); y = self.d0(torch.cat([up(y, x0), x0], 1)); return self.akhir(y)
for nm, m in [("U-Net c=32", UNet()), ("U-Net encoder ResNet-18", UNetResNet18(False))]:
    print(nm, f"{n_param(m)/1e6:.2f} juta parameter", tuple(m(torch.zeros(1, 1, UK, UK)).shape))

## 9.3 *Loss* dan pelatihan
- **BCE** per piksel; **Dice loss** `1 − 2|P∩G| / (|P|+|G|)` (versi lunak, per citra); kombinasi Dice + BCE.
- Augmentasi **bersama** citra–mask: flip horizontal/vertikal, rotasi ±15°, geser ±5 %; kecerahan/kontras ±15 % hanya pada citra.
- AdamW lr 1e-3 (encoder pra-latih: 3e-4), kosinus, maks 40 epoch, *early stopping* pada Dice validasi (patience 8).

In [ ]:
def dice_lunak(lg, m, eps=1.0):
    p = torch.sigmoid(lg); i = (p * m).sum((1, 2, 3)); return 1 - ((2 * i + eps) / (p.sum((1, 2, 3)) + m.sum((1, 2, 3)) + eps)).mean()
LOSS = {"BCE": lambda lg, m: F.binary_cross_entropy_with_logits(lg, m),
        "Dice+BCE": lambda lg, m: F.binary_cross_entropy_with_logits(lg, m) + dice_lunak(lg, m)}
def aug_bersama(x, m):
    n = x.shape[0]; d = x.device
    f = (torch.rand(n, device=d) < 0.5)[:, None, None, None]; x, m = torch.where(f, x.flip(3), x), torch.where(f, m.flip(3), m)
    f = (torch.rand(n, device=d) < 0.5)[:, None, None, None]; x, m = torch.where(f, x.flip(2), x), torch.where(f, m.flip(2), m)
    a = (torch.rand(n, device=d) * 2 - 1) * 15 * np.pi / 180; t = (torch.rand(n, 2, device=d) * 2 - 1) * 0.1
    th = torch.stack([torch.stack([a.cos(), -a.sin(), t[:, 0]], 1), torch.stack([a.sin(), a.cos(), t[:, 1]], 1)], 1)
    gr = F.affine_grid(th, x.shape, align_corners=False)
    x = F.grid_sample(x, gr, padding_mode="zeros", align_corners=False); m = (F.grid_sample(m, gr, mode="nearest", align_corners=False) > 0.5).float()
    mu = x.mean((1, 2, 3), keepdim=True)
    x = ((x - mu) * (1 + (torch.rand(n, 1, 1, 1, device=d) * 2 - 1) * 0.15) + mu) * (1 + (torch.rand(n, 1, 1, 1, device=d) * 2 - 1) * 0.15)
    return x.clamp(0, 1), m
@torch.no_grad()
def prob_semua(model, idx, bs=32):
    model.eval(); out = []
    for i in range(0, len(idx), bs):
        x = XT[idx[i:i + bs]].float().div(255)
        with torch.autocast("cuda", dtype=torch.float16, enabled=AMP): out.append(torch.sigmoid(model(x).float()))
    return torch.cat(out)
def dice_keras(p, m): p = p > 0.5; m = m > 0.5; return ((2 * (p & m).sum((1, 2, 3)) + 1e-6) / (p.sum((1, 2, 3)) + m.sum((1, 2, 3)) + 1e-6)).float()
def latih_seg(nama, model, fungsi_loss, lr=1e-3, maks_epoch=40, patience=8, bs=16, seed=SEED):
    seed_semua(seed); model = model.to(DEV); g = torch.Generator().manual_seed(seed); itr, iv = SPL["train"], SPL["val"]
    if SMOKE: maks_epoch, patience = 2, 1
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4); sch = torch.optim.lr_scheduler.CosineAnnealingLR(opt, maks_epoch)
    scaler = torch.amp.GradScaler("cuda", enabled=AMP); best = (-1, None, 0); log = []; tunggu = 0
    for ep in range(1, maks_epoch + 1):
        model.train(); t0 = time.perf_counter(); perm = itr[torch.randperm(len(itr), generator=g).numpy()]; tl = 0
        for i in range(0, len(perm), bs):
            b = perm[i:i + bs]; x, m = aug_bersama(XT[b].float().div(255), MT[b].float())
            with torch.autocast("cuda", dtype=torch.float16, enabled=AMP): lg = model(x)
            loss = fungsi_loss(lg.float(), m); opt.zero_grad(set_to_none=True); scaler.scale(loss).backward(); scaler.step(opt); scaler.update(); tl += loss.item() * len(b)
        sch.step()
        if DEV.type == "cuda": torch.cuda.synchronize()
        dt = time.perf_counter() - t0; dv = float(dice_keras(prob_semua(model, iv), MT[iv].float()).mean())
        log.append({"model": nama, "epoch": ep, "loss_latih": tl / len(perm), "dice_val": dv, "detik_epoch": dt}); print(log[-1])
        if dv > best[0]: best = (dv, {k: v.detach().clone() for k, v in model.state_dict().items()}, ep); tunggu = 0
        else:
            tunggu += 1
            if tunggu >= patience: print("  early stopping"); break
    model.load_state_dict(best[1]); return model, pd.DataFrame(log), best[2]
KONFIG = {"U-Net (BCE)": (lambda: UNet(), "BCE", 1e-3), "U-Net (Dice+BCE)": (lambda: UNet(), "Dice+BCE", 1e-3),
          "U-Net encoder ResNet-18 pra-latih (Dice+BCE)": (lambda: UNetResNet18(True), "Dice+BCE", 3e-4)}
SEEDS = [0] if SMOKE else [0, 1]
MODEL, LOG = {}, {}
for nk, (fm, nl, lr) in KONFIG.items():
    for s in SEEDS:
        with catat_waktu(f"latih_{nk[:18]}_s{s}"): MODEL[(nk, s)], LOG[(nk, s)], _ = latih_seg(f"{nk} | seed {s}", fm(), LOSS[nl], lr=lr, seed=s)
pd.concat(LOG.values()).to_csv(OUT / "log_epoch.csv", index=False)
fig, ax = plt.subplots(1, 2, figsize=(10, 3.4))
for (nk, s), lg in LOG.items(): ax[0].plot(lg.epoch, lg.loss_latih, label=f"{nk} s{s}"); ax[1].plot(lg.epoch, lg.dice_val, label=f"{nk} s{s}")
ax[0].set_ylabel("loss latih"); ax[1].set_ylabel("Dice val (UK×UK)")
for a in ax: a.set_xlabel("epoch"); a.legend(fontsize=6); a.xaxis.set_major_locator(matplotlib.ticker.MaxNLocator(integer=True))
plt.tight_layout(); plt.savefig(OUT / "gambar_9_1_kurva_latih.png", bbox_inches="tight"); plt.show()

## 9.4 Evaluasi per kasus pada resolusi asli: Dice, IoU, HD95, dan HC dari elips
Probabilitas diperbesar kembali ke 800×540, diambang 0,5, lalu **pascaproses**: komponen terbesar + isi lubang.
HD95 = persentil ke-95 jarak permukaan dua arah, dikali ukuran piksel (mm).

In [ ]:
def pascaproses(b):
    lab = measure.label(b)
    if lab.max() == 0: return b
    return ndi.binary_fill_holes(lab == (np.bincount(lab.ravel())[1:].argmax() + 1))
def hd95(a, b, px):
    if a.sum() == 0 or b.sum() == 0: return np.nan
    ta = a ^ binary_erosion(a); tb = b ^ binary_erosion(b)
    da = distance_transform_edt(~tb)[ta]; db = distance_transform_edt(~ta)[tb]
    return float(np.percentile(np.concatenate([da, db]), 95) * px)
def evaluasi(model):
    P = prob_semua(model, SPL["test"]).cpu(); baris = []
    for k, i in enumerate(SPL["test"]):
        g = M_ASLI[i]; H, W = g.shape
        p = pascaproses(F.interpolate(P[k:k + 1], size=(H, W), mode="bilinear", align_corners=False)[0, 0].numpy() > 0.5)
        it = (p & g).sum(); un = (p | g).sum()
        baris.append({"berkas": META.berkas[i], "dice": 2 * it / (p.sum() + g.sum() + 1e-9), "iou": it / (un + 1e-9), "hd95_mm": hd95(p, g, META.pixel_mm[i]),
                      "hc_acuan_mm": META.hc_mm[i], "hc_elips_pred_mm": keliling_elips_mm(p, META.pixel_mm[i])})
    return pd.DataFrame(baris)
HASIL, PER_KASUS = [], {}
with catat_waktu("4_evaluasi"):
    for (nk, s), m in MODEL.items():
        d = evaluasi(m); d["galat_hc_mm"] = d.hc_elips_pred_mm - d.hc_acuan_mm; PER_KASUS[(nk, s)] = d
        HASIL.append({"konfigurasi": nk, "seed": s, "n_param_juta": n_param(m) / 1e6, "epoch_terbaik": int(LOG[(nk, s)].dice_val.idxmax() + 1),
                      "detik_per_epoch": float(LOG[(nk, s)].detik_epoch.mean()), "dice_val_UK": float(LOG[(nk, s)].dice_val.max()),
                      "dice_uji_rerata": d.dice.mean(), "dice_uji_median": d.dice.median(), "iou_uji_rerata": d.iou.mean(), "hd95_uji_median_mm": d.hd95_mm.median(),
                      "MAE_HC_uji_mm": d.galat_hc_mm.abs().mean(), "bias_HC_uji_mm": d.galat_hc_mm.mean(), "persen_kasus_dice<0.9": float((d.dice < 0.9).mean() * 100)})
        print(HASIL[-1])
T91 = pd.DataFrame(HASIL); T91.to_csv(OUT / "tabel_9_1_per_seed.csv", index=False)
T92 = T91.groupby("konfigurasi")[["dice_uji_rerata", "iou_uji_rerata", "hd95_uji_median_mm", "MAE_HC_uji_mm", "bias_HC_uji_mm", "persen_kasus_dice<0.9", "n_param_juta", "detik_per_epoch"]].agg(["mean", "std"])
T92.columns = ["_".join(c) for c in T92.columns]
T92.loc["acuan: elips dari mask anotasi", "MAE_HC_uji_mm_mean"] = float(np.nanmean(np.abs(HC_GT_ELIPS - META.hc_mm.values[SPL["test"]])))
T92.to_csv(OUT / "tabel_9_2_ringkas.csv"); T92

## 9.5 Visualisasi: sebaran Dice, kasus terbaik/median/terburuk, dan Bland–Altman HC

In [ ]:
PIL = max(MODEL, key=lambda k: T91[(T91.konfigurasi == k[0]) & (T91.seed == k[1])].dice_val_UK.iloc[0]); d = PER_KASUS[PIL]
d.to_csv(OUT / "per_kasus_model_terpilih.csv", index=False); print("model terpilih (Dice val):", PIL)
fig, ax = plt.subplots(figsize=(7, 3.2))
ax.boxplot([PER_KASUS[(nk, SEEDS[0])].dice for nk in KONFIG], tick_labels=[nk.replace(" (", "\n(") for nk in KONFIG], showfliers=True)
ax.set_ylabel("Dice per kasus (uji, resolusi asli)"); plt.tight_layout(); plt.savefig(OUT / "gambar_9_2_boxplot_dice.png", bbox_inches="tight"); plt.show()
urut = d.dice.argsort().values; pilih = {"terburuk": urut[0], "median": urut[len(urut) // 2], "terbaik": urut[-1]}
P = prob_semua(MODEL[PIL], SPL["test"]).cpu()
fig, ax = plt.subplots(1, 3, figsize=(13, 3.6))
for a, (nm, k) in zip(ax, pilih.items()):
    i = SPL["test"][k]; img = np.array(Image.open(AKAR / META.berkas[i]).convert("L")); g = M_ASLI[i]
    p = pascaproses(F.interpolate(P[k:k + 1], size=g.shape, mode="bilinear", align_corners=False)[0, 0].numpy() > 0.5)
    a.imshow(img, cmap="gray"); a.contour(g, [0.5], colors="lime", linewidths=1); a.contour(p, [0.5], colors="red", linewidths=1, linestyles="--")
    a.set_title(f"{nm}: Dice {d.dice.iloc[k]:.3f}, HD95 {d.hd95_mm.iloc[k]:.1f} mm", fontsize=8); a.axis("off")
fig.text(0.5, 0.01, "hijau = anotasi, merah putus-putus = prediksi", ha="center", fontsize=8)
plt.tight_layout(); plt.savefig(OUT / "gambar_9_3_tumpang_susun.png", bbox_inches="tight"); plt.show()
e = d.galat_hc_mm.values; r_ = (d.hc_elips_pred_mm + d.hc_acuan_mm).values / 2; bias = np.nanmean(e); sd = np.nanstd(e, ddof=1)
BA = {"model": f"{PIL[0]} | seed {PIL[1]}", "MAE_HC_mm": float(np.nanmean(np.abs(e))), "bias_mm": float(bias), "LoA_bawah_mm": float(bias - 1.96 * sd), "LoA_atas_mm": float(bias + 1.96 * sd),
      "MAE_HC_CI95": ci_bootstrap(lambda y, p: np.nanmean(np.abs(p - y)), d.hc_acuan_mm.values, d.hc_elips_pred_mm.values, n_boot=200 if SMOKE else 1000, strata=False)}
fig, ax = plt.subplots(1, 2, figsize=(10, 3.6))
ax[0].scatter(d.hc_acuan_mm, d.hc_elips_pred_mm, s=8, alpha=0.6); lim = [d.hc_acuan_mm.min() - 10, d.hc_acuan_mm.max() + 10]; ax[0].plot(lim, lim, "k--", lw=1)
ax[0].set_xlabel("HC acuan (mm)"); ax[0].set_ylabel("HC dari elips prediksi (mm)"); ax[0].set_title(f"MAE {BA['MAE_HC_mm']:.2f} mm")
ax[1].scatter(r_, e, s=8, alpha=0.6); ax[1].axhline(bias, color="r")
for v in (BA["LoA_bawah_mm"], BA["LoA_atas_mm"]): ax[1].axhline(v, color="r", ls="--")
ax[1].set_xlabel("rerata (mm)"); ax[1].set_ylabel("elips prediksi − acuan (mm)"); ax[1].set_title(f"Bland–Altman: bias {bias:.2f}, LoA [{BA['LoA_bawah_mm']:.1f}, {BA['LoA_atas_mm']:.1f}] mm")
plt.tight_layout(); plt.savefig(OUT / "gambar_9_4_hc_dari_mask.png", bbox_inches="tight"); plt.show()
torch.save(MODEL[PIL].state_dict(), OUT / "unet_hc18_terpilih.pt"); BA

## 9.6 Ringkasan, zip hasil, dan total waktu

In [ ]:
def bersih(o):
    if isinstance(o, dict): return {str(k): bersih(v) for k, v in o.items()}
    if isinstance(o, (list, tuple)): return [bersih(v) for v in o]
    if isinstance(o, (np.floating, float)): return round(float(o), 6)
    if isinstance(o, (np.integer,)): return int(o)
    if isinstance(o, np.bool_): return bool(o)
    return o
WAKTU["total_notebook"] = round(time.perf_counter() - T_MULAI, 2)
GPU = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "tidak ada GPU (CPU)"
RINGKASAN = bersih(dict(smoke=SMOKE, gpu=GPU, versi=VERSI, waktu_detik=WAKTU, **(dict(ukuran_px=UK, split={s: len(i) for s, i in SPL.items()}, per_seed=T91.to_dict("records"), elips_mask_acuan_MAE_mm=float(np.nanmean(np.abs(HC_GT_ELIPS - META.hc_mm.values[SPL["test"]]))), model_terpilih=list(PIL), bland_altman_hc=BA))))
(OUT / "ringkasan_bab09.json").write_text(json.dumps(RINGKASAN, indent=1, ensure_ascii=False))
pd.Series(WAKTU, name="detik").to_csv(OUT / "waktu_sel.csv")
arsip = shutil.make_archive(str(OUT), "zip", root_dir=OUT)
print("Berkas hasil:"); [print(" -", p.name, f"{p.stat().st_size/1e3:.0f} kB") for p in sorted(OUT.iterdir())]
print("ZIP:", arsip, f"{Path(arsip).stat().st_size/1e6:.2f} MB")
print(json.dumps(WAKTU, indent=1))
print(f"TOTAL WAKTU NOTEBOOK: {WAKTU['total_notebook']:.1f} s ({WAKTU['total_notebook']/60:.1f} menit) | GPU: {GPU}")
try:
    from google.colab import files  # unduh otomatis bila di Colab
    files.download(arsip)
except Exception:
    pass